# Agent 4 — Document Intelligence

**What this notebook does:**  
Imports and organises the structured data extracted by Claude Projects (your RAG system) from company sustainability reports, TCFD filings, and CSRD/ESRS documents. It merges those AI-extracted insights with the quantitative master dataset.

**How to present this to investors:**  
> *Our document intelligence agent processes 15–20 primary source PDFs through a retrieval-augmented generation pipeline. For each company, Claude Projects extracts verbatim quotes, page numbers, and structured data points from sustainability reports. The results are imported here as verified, citation-backed data — distinguishing reported figures from AI estimates throughout.*

**How this workflow operates:**  
1. RAG Operator uploads company PDFs to Claude Projects (one project per company or corpus)
2. RAG Operator runs the Document Intelligence prompt (below) in each Claude Project
3. RAG Operator copies the JSON response and saves it as a `.json` file in `outputs/rag/`
4. This notebook imports all those JSON files and merges them into the master dataset

**Run after:** `01_data_ingestion.ipynb` and `01b_data_quality.ipynb`

## Step 1 — The Document Intelligence RAG prompt

Copy this prompt into each Claude Project and run it against each company's PDF corpus.  
Save the JSON output as `outputs/rag/doc_intel_[TICKER].json`.

In [1]:
RAG_PROMPT = """
You are a sustainable finance analyst. Using ONLY the documents in this project, 
extract the following information for [COMPANY NAME / TICKER].

For each field: provide (a) the verbatim quote, (b) the page number or section,
(c) the document title and year, and (d) confidence (HIGH / MEDIUM / LOW / MISSING).
If information is absent, mark as MISSING — never invent or infer.

Output ONLY valid JSON with this structure:
{
  "ticker": "...",
  "company_name": "...",
  "extraction_date": "YYYY-MM-DD",
  "fields": {
    "scope1_emissions":       {"value": null, "unit": null, "year": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "scope2_emissions":       {"value": null, "unit": null, "year": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "scope3_emissions":       {"value": null, "unit": null, "year": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "net_zero_target_year":   {"value": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "sbti_status":            {"value": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "revenue_green_pct":      {"value": null, "unit": "%", "quote": null, "page": null, "doc": null, "confidence": null},
    "eu_taxonomy_eligible":   {"value": null, "unit": "%", "quote": null, "page": null, "doc": null, "confidence": null},
    "eu_taxonomy_aligned":    {"value": null, "unit": "%", "quote": null, "page": null, "doc": null, "confidence": null},
    "board_gender_diversity": {"value": null, "unit": "% women", "quote": null, "page": null, "doc": null, "confidence": null},
    "biodiversity_risk_flag": {"value": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "water_withdrawal":       {"value": null, "unit": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "controversy_flag":       {"value": null, "quote": null, "page": null, "doc": null, "confidence": null},
    "assurance_provider":     {"value": null, "quote": null, "page": null, "doc": null, "confidence": null}
  }
}
"""

print("Document Intelligence RAG prompt ready.")
print("")
print("INSTRUCTIONS FOR RAG OPERATOR:")
print("1. Open a Claude Project for the company you are analysing")
print("2. Make sure the company's PDFs are uploaded to that project")
print("3. Replace [COMPANY NAME / TICKER] with the actual company")
print("4. Paste the prompt above into the chat and run it")
print("5. Copy the JSON output")
print("6. Save it as:  outputs/rag/doc_intel_TICKER.json")
print("   Example:     outputs/rag/doc_intel_ASML.json")
print("")
print("Then come back and run the rest of this notebook.")

Document Intelligence RAG prompt ready.

INSTRUCTIONS FOR RAG OPERATOR:
1. Open a Claude Project for the company you are analysing
2. Make sure the company's PDFs are uploaded to that project
3. Replace [COMPANY NAME / TICKER] with the actual company
4. Paste the prompt above into the chat and run it
5. Copy the JSON output
6. Save it as:  outputs/rag/doc_intel_TICKER.json
   Example:     outputs/rag/doc_intel_ASML.json

Then come back and run the rest of this notebook.


## Step 2 — Create the output folder and a sample JSON file

Run this cell once to create the folder structure.  
The sample file shows the exact format your JSON outputs should follow.

In [2]:
import os
import json
import pandas as pd
import glob
from datetime import date

TODAY = str(date.today())
RAG_DIR = "../outputs/rag"
os.makedirs(RAG_DIR, exist_ok=True)

# Write a sample/template file so the RAG Operator knows the expected format
sample = {
  "ticker": "EXAMPLE",
  "company_name": "Example Company SA",
  "extraction_date": TODAY,
  "fields": {
    "scope1_emissions":       {"value": 1250000, "unit": "tCO2e", "year": 2023, "quote": "Direct GHG emissions totalled 1,250,000 tCO2e in FY2023.", "page": 42, "doc": "Sustainability Report 2023", "confidence": "HIGH"},
    "scope2_emissions":       {"value": 340000,  "unit": "tCO2e", "year": 2023, "quote": "Market-based Scope 2 emissions were 340,000 tCO2e.",          "page": 42, "doc": "Sustainability Report 2023", "confidence": "HIGH"},
    "scope3_emissions":       {"value": None, "unit": None, "year": None, "quote": None, "page": None, "doc": None, "confidence": "MISSING"},
    "net_zero_target_year":   {"value": 2040, "quote": "We commit to net zero by 2040 across all scopes.", "page": 8, "doc": "Sustainability Report 2023", "confidence": "HIGH"},
    "sbti_status":            {"value": "Committed", "quote": "Example Company signed the SBTi Business Ambition in 2022.", "page": 9, "doc": "Sustainability Report 2023", "confidence": "MEDIUM"},
    "revenue_green_pct":      {"value": 38.0, "unit": "%", "quote": "38% of revenues are derived from low-carbon products.", "page": 15, "doc": "Annual Report 2023", "confidence": "MEDIUM"},
    "eu_taxonomy_eligible":   {"value": 52.0, "unit": "%", "quote": "52.3% of turnover is taxonomy-eligible.", "page": 88, "doc": "Annual Report 2023", "confidence": "HIGH"},
    "eu_taxonomy_aligned":    {"value": None, "unit": "%", "quote": None, "page": None, "doc": None, "confidence": "MISSING"},
    "board_gender_diversity": {"value": 42.0, "unit": "% women", "quote": "42% of board seats are held by women.", "page": 60, "doc": "Annual Report 2023", "confidence": "HIGH"},
    "biodiversity_risk_flag": {"value": "Low", "quote": "No material operations adjacent to protected areas.", "page": 50, "doc": "Sustainability Report 2023", "confidence": "LOW"},
    "water_withdrawal":       {"value": 2100000, "unit": "m3", "quote": "Total water withdrawal: 2.1 million m3.", "page": 48, "doc": "Sustainability Report 2023", "confidence": "HIGH"},
    "controversy_flag":       {"value": "None identified", "quote": None, "page": None, "doc": None, "confidence": "LOW"},
    "assurance_provider":     {"value": "Deloitte", "quote": "Assured by Deloitte to limited assurance standard.", "page": 95, "doc": "Sustainability Report 2023", "confidence": "HIGH"}
  }
}

template_path = f"{RAG_DIR}/doc_intel_EXAMPLE_TEMPLATE.json"
with open(template_path, "w") as f:
    json.dump(sample, f, indent=2)

print(f"Template saved: {template_path}")
print(f"RAG output folder: {RAG_DIR}")
print("\nSave your Claude Projects JSON outputs here as: doc_intel_TICKER.json")

Template saved: ../outputs/rag/doc_intel_EXAMPLE_TEMPLATE.json
RAG output folder: ../outputs/rag

Save your Claude Projects JSON outputs here as: doc_intel_TICKER.json


In [ ]:
# ── Corpus inventory — RAG source documents available in the project ──────────
# data/rag/corpus/ holds the sustainability / annual-report PDFs, one folder per
# capped-Top-40 company. NB06 scores from the doc_intel JSON *outputs* of the RAG
# step — not the PDFs themselves — so this cell reports corpus coverage: how many
# companies still need a Claude Projects extraction run.

CORPUS_DIR = "../data/rag/corpus"
corpus = sorted(d for d in glob.glob(f"{CORPUS_DIR}/*") if os.path.isdir(d))
n_pdf  = len(glob.glob(f"{CORPUS_DIR}/**/*.pdf", recursive=True))
n_json = len([f for f in glob.glob(f"{RAG_DIR}/doc_intel_*.json") if "TEMPLATE" not in f])

print(f"RAG corpus ({CORPUS_DIR}):  {len(corpus)} company folders,  {n_pdf} source PDFs")
print(f"Doc-intel JSON extractions in {RAG_DIR}:  {n_json}")
print("Workflow: upload a company's PDFs to its Claude Project -> run the Step 1")
print("prompt -> save outputs/rag/doc_intel_<TICKER>.json -> re-run Step 3 below.")
print()
if corpus:
    for d in corpus:
        npdf = len(glob.glob(f"{d}/**/*.pdf", recursive=True))
        print(f"  {os.path.basename(d):<44} {npdf:>2} PDF(s)")
else:
    print("  Corpus not found at data/rag/corpus/ — see data/rag/corpus/README.md")

## Step 3 — Import all RAG JSON files

Once the RAG Operator has saved JSON files for each company, run this cell to load them all.

In [3]:
# Find all doc_intel JSON files (excluding the template)
rag_files = [f for f in glob.glob(f"{RAG_DIR}/doc_intel_*.json") if "TEMPLATE" not in f]

if not rag_files:
    print("No RAG output files found yet.")
    print(f"Save your Claude Projects JSON outputs to: {RAG_DIR}/doc_intel_TICKER.json")
    print("Then re-run this cell.")
else:
    print(f"Found {len(rag_files)} RAG output files:")
    for f in rag_files:
        print(f"  {os.path.basename(f)}")

No RAG output files found yet.
Save your Claude Projects JSON outputs to: ../outputs/rag/doc_intel_TICKER.json
Then re-run this cell.


In [4]:
# Parse all JSON files into a flat DataFrame
records = []
for fpath in rag_files:
    with open(fpath) as f:
        data = json.load(f)
    
    row = {
        "ticker": data.get("ticker"),
        "company_name": data.get("company_name"),
        "extraction_date": data.get("extraction_date")
    }
    
    for field, info in data.get("fields", {}).items():
        row[f"rag_{field}"]            = info.get("value")
        row[f"rag_{field}_confidence"] = info.get("confidence")
        row[f"rag_{field}_quote"]      = info.get("quote")
        row[f"rag_{field}_page"]       = info.get("page")
        row[f"rag_{field}_doc"]        = info.get("doc")
    
    records.append(row)

if records:
    rag_df = pd.DataFrame(records)
    print(f"RAG data imported: {len(rag_df)} companies x {len(rag_df.columns)} columns")
    print("\nCompanies in RAG dataset:")
    print(rag_df[["ticker", "company_name"]].to_string(index=False))
    rag_df.head(3)
else:
    rag_df = pd.DataFrame()
    print("No records to show — add JSON files and re-run.")

No records to show — add JSON files and re-run.


## Step 4 — Confidence summary

Shows how many fields have HIGH / MEDIUM / LOW / MISSING confidence across all companies.

In [5]:
if not rag_df.empty:
    conf_cols = [c for c in rag_df.columns if c.endswith("_confidence")]
    conf_values = rag_df[conf_cols].values.flatten()
    conf_series = pd.Series(conf_values).dropna()
    summary = conf_series.value_counts()
    total = len(conf_series)
    print("RAG extraction confidence breakdown:")
    for level in ["HIGH", "MEDIUM", "LOW", "MISSING"]:
        count = summary.get(level, 0)
        pct = count / total * 100 if total > 0 else 0
        print(f"  {level:<8} {count:>4} fields ({pct:.1f}%)")
    print(f"\n  Total fields assessed: {total}")
    print(f"\nVerification requirement:")
    n_verify = max(1, round(total * 0.30))
    print(f"  30% random sample = {n_verify} field extractions must be manually checked")
    print(f"  100% of fields driving exclusion/watchlist decisions must be verified")
else:
    print("No RAG data loaded yet.")

No RAG data loaded yet.


## Step 5 — Save and merge with master dataset

In [6]:
if not rag_df.empty:
    # Save standalone RAG table
    rag_out = f"../outputs/scores/rag_extractions_{TODAY}.csv"
    rag_df.to_csv(rag_out, index=False)
    print(f"RAG extractions saved: {rag_out}")

    # Optionally merge with master dataset
    master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
    if master_files:
        master = pd.read_csv(master_files[-1])
        TICKER_COLUMN = "ticker"  # update if your ticker column has a different name
        if TICKER_COLUMN in master.columns and "ticker" in rag_df.columns:
            merged = master.merge(rag_df.drop(columns=["company_name", "extraction_date"], errors="ignore"),
                                  left_on=TICKER_COLUMN, right_on="ticker", how="left")
            merged_out = f"../outputs/scores/master_with_rag_{TODAY}.csv"
            merged.to_csv(merged_out, index=False)
            print(f"Merged with master: {merged_out}")
            print(f"Shape: {merged.shape}")
        else:
            print("Could not merge — check TICKER_COLUMN name")
    else:
        print("Master dataset not found — run 01_data_ingestion.ipynb first")
else:
    print("No RAG data to save yet.")

No RAG data to save yet.


## ✅ Notebook complete

You now have:
- A **structured RAG output table** with citation-backed data points per company
- A **confidence breakdown** showing how many fields are HIGH / MISSING
- A **merged master dataset** combining quantitative CSV data with AI-extracted document intelligence

**Verification reminder:** The RAG Operator must manually verify 30% of extractions against the source PDF (with page number). All extractions that feed exclusion or watchlist decisions must be 100% verified.

**Next:** Open `07_biodiversity.ipynb` to add nature-risk proxy scores.